# Assignment 2: From Trees to Neural Networks — Step 1 (learning notebook)

**Dataset:** UCI Bank Marketing (`bank-additional-full.csv`)

> ## How to use this notebook
> Same code as the final Step 1 notebook, plus learning notes, self-check questions, and report prompts.
> In Colab: upload this `.ipynb`, upload `bank-additional-full.csv` to the working directory, then Run all.
> The **final** notebook is the graded Colab artifact; this one is for study and report prep.

## 0. Setup

> **Learning note — what the constants mean**
> - `RANDOM_STATE = 42`: the handout requires this seed so the split is reproducible.
> - `LEAKAGE_COLUMNS = ["duration"]`: kept as a named constant so the exclusion is explicit and easy to check.
> - `PDAYS_NO_CONTACT = 999` / `PDAYS_NO_CONTACT_ENCODED = -1`: the sentinel and what we replace it with.
> - Versions and CPU info are printed because Step 4 must report hardware and package versions alongside timings.

In [ ]:
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_FILE = "bank-additional-full.csv"
TARGET = "y"
POSITIVE_LABEL = "yes"
LEAKAGE_COLUMNS = ["duration"]
PDAYS_NO_CONTACT = 999
PDAYS_NO_CONTACT_ENCODED = -1

print(f"Python:       {platform.python_version()}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Platform:     {platform.platform()}")
print(f"CPU cores:    {os.cpu_count()}")
print(f"RANDOM_STATE: {RANDOM_STATE}")

### 0.1 Load the data

Place `bank-additional-full.csv` in the same working directory as this notebook. Source: UCI Machine Learning Repository, Bank Marketing dataset — `bank-additional-full.csv` from `bank-additional.zip`. The file is semicolon-separated.

In [ ]:
DATA_PATH = Path(DATA_FILE)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} not found in the working directory. "
        "In Colab: Files → Upload → select bank-additional-full.csv, then re-run this cell."
    )

print(f"Found: {DATA_PATH.resolve()}")

## 1. Data audit

> **Learning note — why audit first?**
> Step 1a asks for row count, feature types, positive-class proportion, and missing/special values, *and* a justification of cleaning choices. You can only justify a choice after looking at the data.

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")
df.head()

### 1.1 Feature types

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols and c != TARGET]

feature_types = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
}).drop(index=TARGET)
feature_types["type"] = np.where(feature_types.index.isin(numeric_cols), "numeric", "nominal categorical")

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
feature_types

> **Learning note — numeric vs nominal**
> *Nominal* means categories with no natural order (`job`, `month`, …). One-hot encoding turns each level into its own 0/1 column so the model doesn't assume `admin.` < `blue-collar`.
> `education` has a rough order, but its `"unknown"` and `"illiterate"` levels don't fit cleanly into a ranking, so we treat it as nominal too.

### 1.2 Target distribution

In [ ]:
target_counts = df[TARGET].value_counts()
target_share = df[TARGET].value_counts(normalize=True)
pd.DataFrame({"count": target_counts, "proportion": target_share.round(4)})

> **Learning note — class imbalance and why AP is the main metric**
> Only ~11% are positive. A model that always says "no" gets ~89% accuracy and is useless.
> **Average Precision (AP)** summarizes the precision–recall curve for the positive class. Its "no-skill" baseline equals the positive rate (~0.113), which is why the Step 4 PR plot needs that reference line.
>
> **Self-check:** What AP would a random classifier get on this data?

### 1.3 Missing and special values

In [ ]:
# pandas nulls vs. the dataset's literal "unknown" category
null_counts = df.isna().sum()
print(f"Total pandas-null cells: {int(null_counts.sum())}")

unknown_summary = pd.DataFrame({
    "unknown_count": [(df[c] == "unknown").sum() for c in categorical_cols],
}, index=categorical_cols)
unknown_summary["unknown_pct"] = (100 * unknown_summary["unknown_count"] / len(df)).round(2)
unknown_summary.sort_values("unknown_count", ascending=False)

> **Learning note — three different kinds of "missing"**
> 1. **NaN** (pandas null): an empty cell. This dataset has none.
> 2. **Literal `"unknown"`**: the bank recorded that the value was unknown. It is a real category.
> 3. **Unseen category**: a value that appears in validation/test but never in training. `handle_unknown="ignore"` deals with this.
> The handout asks you to explain (2) and (3) separately.

In [ ]:
# Positive rate when a field is "unknown" vs. known: is "unknown" informative?
rows = []
for col in unknown_summary.index[unknown_summary["unknown_count"] > 0]:
    is_unknown = df[col] == "unknown"
    rows.append({
        "column": col,
        "positive_rate_unknown": (df.loc[is_unknown, TARGET] == POSITIVE_LABEL).mean(),
        "positive_rate_known": (df.loc[~is_unknown, TARGET] == POSITIVE_LABEL).mean(),
    })
pd.DataFrame(rows).set_index("column").round(4)

> **Learning note — is "unknown" informative?**
> Compare the two rates per column. If they differ a lot (as for `default`), "unknown" carries signal, so imputing it with the most common value would erase useful information. That's the argument for keeping it as its own category.

In [ ]:
no_contact = df["pdays"] == PDAYS_NO_CONTACT
print(f"pdays == {PDAYS_NO_CONTACT}: {no_contact.sum():,} rows ({no_contact.mean():.2%})")
print("pdays range when a previous contact exists:",
      df.loc[~no_contact, "pdays"].min(), "to", df.loc[~no_contact, "pdays"].max())
print()
print(pd.crosstab(no_contact.rename("pdays == 999"), df["poutcome"]))

> **Learning note — the `pdays = 999` sentinel**
> 999 does **not** mean 999 days; it means "never contacted before". Left as-is:
> - an MLP sees one huge value that dominates the scaled column;
> - a linear-ish reading would treat "never contacted" as "contacted very long ago".
> Our fix: an explicit `previously_contacted` 0/1 flag plus `pdays = -1` for no contact. Trees can split at `pdays < 0`; the MLP gets the flag directly.
> The crosstab shows consistency with `poutcome`: `nonexistent` ⇔ 999.

In [ ]:
print(f"Exact duplicate rows (all columns):           {df.duplicated().sum():,}")
print(f"Duplicate rows after excluding duration:      {df.drop(columns=LEAKAGE_COLUMNS).duplicated().sum():,}")
print(f"Rows where default == 'yes':                  {(df['default'] == 'yes').sum():,}")

> **Learning note — duplicates**
> 12 exact duplicates is tiny. Without an ID we can't tell if they're the same client recorded twice. Dropping `duration` makes 1,784 rows identical, which just shows the remaining features are coarse (many clients share job, month, and macro-indicator values). Those are different calls, not data errors, so we keep them.
>
> **Self-check:** Why would de-duplicating *after* dropping `duration` be a bad idea?

**Audit findings (fill from your Colab output for the report).**

- Rows / columns: 41,188 × 21 (10 numeric, 10 nominal categorical, target `y`)
- Positive rate: ~11.3%
- NaNs: none; literal `"unknown"` in 6 columns (largest: `default`, ~20.9%)
- `pdays = 999`: ~96.3% of rows; real range 0–27
- Exact duplicates: 12

## 2. Cleaning decisions and target definition

| Item | Decision | Reason |
|---|---|---|
| Target | `y = "yes"` → 1 | Handout positive class |
| `duration` | Dropped | Known only after the call → leakage |
| `pdays = 999` | `previously_contacted` flag + `pdays = -1` | 999 is a sentinel, not a duration |
| `"unknown"` | Own one-hot category | Recorded value with its own signal |
| NaN | No imputation | None present |
| Duplicates | Kept | 12 rows, no ID to confirm |
| `default = "yes"` | Kept | 3 rows; ignored if absent in train |

> **Learning note — why is it OK to clean before splitting?**
> Leakage happens when a step *learns statistics* (a median, a category list, a mean/std) from rows it shouldn't see. Mapping 999 → -1, creating a flag, and dropping a column are fixed rules applied row by row; they learn nothing. Anything learned (encoder categories, scaler mean/std) is fitted after the split, on train only.
>
> **Report prompt (Methods):** In 2–3 sentences, explain why `duration` is excluded and how `pdays = 999` and `"unknown"` are represented.

In [ ]:
def prepare_features(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Deterministic, row-wise cleaning only (no statistics learned from the data)."""
    data = raw.copy()
    y = (data[TARGET] == POSITIVE_LABEL).astype(int).rename("target")

    data["previously_contacted"] = (data["pdays"] != PDAYS_NO_CONTACT).astype(int)
    data["pdays"] = data["pdays"].where(data["previously_contacted"] == 1, PDAYS_NO_CONTACT_ENCODED)

    X = data.drop(columns=[TARGET, *LEAKAGE_COLUMNS])
    return X, y


X, y = prepare_features(df)

assert "duration" not in X.columns
assert TARGET not in X.columns
assert (X.loc[X["previously_contacted"] == 0, "pdays"] == PDAYS_NO_CONTACT_ENCODED).all()

print(f"X: {X.shape}  y: {y.shape}  positive rate: {y.mean():.4f}")
X[["pdays", "previously_contacted", "previous", "poutcome"]].head()

In [ ]:
CATEGORICAL_FEATURES = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "day_of_week", "poutcome",
]
NUMERIC_FEATURES = [
    "age", "campaign", "pdays", "previous", "previously_contacted",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
]

assert set(CATEGORICAL_FEATURES) | set(NUMERIC_FEATURES) == set(X.columns)
assert not set(CATEGORICAL_FEATURES) & set(NUMERIC_FEATURES)
print(f"{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical = {X.shape[1]} predictors")

> **Learning note — explicit feature lists**
> Listing columns by hand (instead of selecting by dtype) makes it obvious which columns are scaled and which are encoded, and the asserts guarantee no column is forgotten or counted twice.

## 3. Stratified 70/15/15 train / validation / test split

> **Learning note — why three splits, and why two stages?**
> `train_test_split` makes one cut at a time. First 70/30, then split the 30% in half (`test_size = 0.15 / 0.30 = 0.5`).
> - **Train:** fit preprocessing + models
> - **Validation:** early stopping (XGBoost) and choosing configs by validation AP
> - **Test:** touched once at the very end
> `stratify=` keeps the ~11.3% positive rate in every split. Without it, a small split could have noticeably more or fewer positives just by chance.

In [ ]:
TRAIN_SIZE, VALID_SIZE, TEST_SIZE = 0.70, 0.15, 0.15

# Stage 1: 70% train vs. 30% holdout. Stage 2: split the holdout 50/50 into validation and test.
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, train_size=TRAIN_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_hold, y_hold, test_size=TEST_SIZE / (VALID_SIZE + TEST_SIZE), stratify=y_hold, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "share": [len(y_train) / len(y), len(y_valid) / len(y), len(y_test) / len(y)],
    "positives": [y_train.sum(), y_valid.sum(), y_test.sum()],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"]).round(4)
split_summary

In [ ]:
train_idx, valid_idx, test_idx = set(X_train.index), set(X_valid.index), set(X_test.index)

assert not train_idx & valid_idx and not train_idx & test_idx and not valid_idx & test_idx
assert len(train_idx | valid_idx | test_idx) == len(X)
assert X_train.index.equals(y_train.index) and X_valid.index.equals(y_valid.index) and X_test.index.equals(y_test.index)
print("Splits are disjoint, cover every row, and X/y indices are aligned.")

> **Self-check**
> 1. Why must the test set stay untouched until both final models are chosen?
> 2. What would go wrong if GBDT and MLP used different splits?
>
> **Discussion limitation:** a random split mixes all campaigns together, so it does **not** show how the model would do on *future* campaigns. The handout also calls this a *historical benchmark*: don't present scores as evidence of current banking performance.

## 4. Leakage-safe preprocessing

> **Learning note — `fit_transform` vs `transform`**
> - `preprocessor.fit_transform(X_train)` **learns** (category levels, scaler mean/std) from train and applies them.
> - `preprocessor.transform(X_valid / X_test)` **reuses** what was learned; no refitting.
> If the scaler were fitted on all rows, validation and test statistics would leak into training.
>
> **Learning note — why scale for the MLP but not the trees?**
> Trees split on thresholds (`age < 35`), which only depend on order, so scaling changes nothing. Neural nets use gradient descent on weighted sums; features with very different scales (5,000 vs 0/1) make optimization slow or unstable. The handout says to scale for the MLP and lets trees skip it.
>
> **Learning note — "same input information"**
> Both models get the same rows and the same 63 columns. Scaling is a model-specific *representation* change, not new information.

In [ ]:
def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    numeric_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer(
        transformers=[
            ("num", numeric_step, NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


def fit_transform_splits(preprocessor: ColumnTransformer):
    """Fit on training rows only; reuse the fitted transformer for validation and test."""
    start = time.perf_counter()
    train_t = preprocessor.fit_transform(X_train)
    valid_t = preprocessor.transform(X_valid)
    test_t = preprocessor.transform(X_test)
    return train_t, valid_t, test_t, time.perf_counter() - start


tree_preprocessor = make_preprocessor(scale_numeric=False)
mlp_preprocessor = make_preprocessor(scale_numeric=True)

X_train_tree, X_valid_tree, X_test_tree, tree_preprocess_sec = fit_transform_splits(tree_preprocessor)
X_train_mlp, X_valid_mlp, X_test_mlp, mlp_preprocess_sec = fit_transform_splits(mlp_preprocessor)

PREPROCESSING_TIME_SEC = {"GBDT (one-hot only)": tree_preprocess_sec, "MLP (one-hot + scaling)": mlp_preprocess_sec}

pd.DataFrame({
    "train": [X_train_tree.shape, X_train_mlp.shape],
    "validation": [X_valid_tree.shape, X_valid_mlp.shape],
    "test": [X_test_tree.shape, X_test_mlp.shape],
    "preprocessing_sec": [round(tree_preprocess_sec, 4), round(mlp_preprocess_sec, 4)],
}, index=["GBDT input", "MLP input"])

In [ ]:
# 1. Both models see the same encoded columns (same underlying information).
assert list(X_train_tree.columns) == list(X_train_mlp.columns)

# 2. No missing values after preprocessing.
for frame in [X_train_tree, X_valid_tree, X_test_tree, X_train_mlp, X_valid_mlp, X_test_mlp]:
    assert not frame.isna().any().any()

# 3. Tree numeric columns are unchanged; MLP numeric columns are standardized with training statistics.
assert np.allclose(X_train_tree[NUMERIC_FEATURES].to_numpy(), X_train[NUMERIC_FEATURES].to_numpy())

scaling_check = pd.DataFrame({
    "train_mean": X_train_mlp[NUMERIC_FEATURES].mean(),
    "train_std": X_train_mlp[NUMERIC_FEATURES].std(ddof=0),
    "valid_mean": X_valid_mlp[NUMERIC_FEATURES].mean(),
    "test_mean": X_test_mlp[NUMERIC_FEATURES].mean(),
}).round(4)

n_onehot = len(X_train_tree.columns) - len(NUMERIC_FEATURES)
print(f"Encoded features: {X_train_tree.shape[1]} ({len(NUMERIC_FEATURES)} numeric + {n_onehot} one-hot)")
print("Categories with no training rows (ignored if seen later):",
      [f"{col}={val}" for col in CATEGORICAL_FEATURES for val in X[col].unique()
       if val not in set(X_train[col])] or "none")
scaling_check

> **Learning note — reading the scaling table**
> Train mean ≈ 0, std ≈ 1 → the scaler learned from train. Validation/test means are close to but not exactly 0 → it was **not** refit on them. That's the evidence of no leakage you can quote in review.
>
> **Report prompts**
> - Methods: one-hot (+ `handle_unknown="ignore"`), MLP-only standard scaling, fit on train only.
> - Results (Step 4): quote `PREPROCESSING_TIME_SEC` separately from model fit time.
>
> **TODO before Step 2**
> - [ ] Run all in Colab; confirm split sizes and positive rates
> - [ ] Note the encoded feature count for the report
> - [ ] Be ready to explain: `duration` leakage, `pdays` handling, `"unknown"` handling, why only the MLP is scaled